# Predicting Moral Values in Text
### This Code offers predicting moral values from the MoralBERT weights deployad in Hugging Face.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoConfig,
    PretrainedConfig,
    PreTrainedModel
)
from sklearn.utils import resample
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    multilabel_confusion_matrix as mcm,
    precision_score,
    recall_score
)
from transformers.modeling_outputs import SequenceClassifierOutput

In [3]:
import os
import random
import re
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from itertools import product
from scipy.special import softmax
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    multilabel_confusion_matrix as mcm,
    precision_score,
    recall_score
)
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold, StratifiedKFold, train_test_split
from sklearn.utils import resample
from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight
from tabulate import tabulate
from torch.autograd import Function
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, RandomSampler, SequentialSampler, TensorDataset
from tqdm import trange
from tqdm.auto import trange
from transformers import (
    AutoConfig,
    AutoModel,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BertModel,
    PretrainedConfig,
    PreTrainedModel,
    get_linear_schedule_with_warmup
)
from transformers.modeling_outputs import SequenceClassifierOutput

In [4]:
base_model = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(base_model)

HF_EXPORT_ROOT = Path("/content/drive/MyDrive/moralbert_inggris")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [5]:
from transformers import PretrainedConfig, PreTrainedModel, AutoModel, AutoConfig
from transformers.modeling_outputs import SequenceClassifierOutput
import torch
import torch.nn as nn

class PlainBERTConfig(PretrainedConfig):
    model_type = "plainbert"

    def __init__(
        self,
        base_model_name_or_path="bert-base-uncased",
        num_labels=2,
        class_weight=None,
        identity_weight=0.0,
        reconstruction_weight=0.0,
        moral_weight=1.0,
        freeze_bert=False,
        id2label=None,
        label2id=None,
        **kwargs,
    ):
        super().__init__(num_labels=num_labels, id2label=id2label, label2id=label2id, **kwargs)
        self.base_model_name_or_path = base_model_name_or_path
        self.num_labels = num_labels
        self.class_weight = class_weight if class_weight is not None else [1.0] * num_labels
        self.identity_weight = float(identity_weight)
        self.reconstruction_weight = float(reconstruction_weight)
        self.moral_weight = float(moral_weight)
        self.freeze_bert = bool(freeze_bert)
        self.problem_type = "single_label_classification"


class PlainBERTForSequenceClassification(PreTrainedModel):
    config_class = PlainBERTConfig
    base_model_prefix = "bert"
    supports_gradient_checkpointing = False

    def __init__(self, config):
        super().__init__(config)

        self.num_labels = config.num_labels
        self.freeze = config.freeze_bert

        base_encoder_config = AutoConfig.from_pretrained(config.base_model_name_or_path)
        self.bert = AutoModel.from_config(base_encoder_config)
        bert_dim = self.bert.config.hidden_size

        self.invariant_trans = nn.Linear(bert_dim, bert_dim)

        if config.identity_weight + config.reconstruction_weight == 0:
            self.moral_classification = nn.Linear(bert_dim, config.num_labels)
        else:
            self.moral_classification = nn.Sequential(
                nn.Linear(bert_dim, bert_dim),
                nn.ReLU(),
                nn.Linear(bert_dim, config.num_labels),
            )

        class_weight = config.class_weight if config.class_weight is not None else [1.0] * config.num_labels
        if class_weight and class_weight[0] > 0:
            weights = torch.tensor(class_weight).float()
        else:
            weights = torch.ones(config.num_labels).float()
        self.register_buffer("class_weights", weights)

        self.reconstruction_feed = nn.Linear(bert_dim, bert_dim)
        self.loss_reconstruction = nn.MSELoss()
        self.register_buffer("identity", torch.eye(bert_dim))

        try:
            self.post_init()
        except Exception as e:
            print("Warning during post_init:", e)

    def forward(
        self,
        input_ids=None,
        token_type_ids=None,
        attention_mask=None,
        labels=None,
        original_bert_embeddings=None,
        **kwargs,
    ):
        if self.freeze:
            with torch.no_grad():
                cls = self.bert(
                    input_ids=input_ids,
                    token_type_ids=token_type_ids,
                    attention_mask=attention_mask
                ).last_hidden_state[:, 0, :]
        else:
            cls = self.bert(
                input_ids=input_ids,
                token_type_ids=token_type_ids,
                attention_mask=attention_mask
            ).last_hidden_state[:, 0, :]

        z = self.invariant_trans(cls)
        logits = self.moral_classification(z)

        total_loss = None
        if labels is not None:
            loss_fn_moral = nn.CrossEntropyLoss(weight=self.class_weights)
            loss_moral = loss_fn_moral(logits, labels)

            if original_bert_embeddings is not None and self.config.reconstruction_weight > 0:
                loss_recon = self.loss_reconstruction(
                    self.reconstruction_feed(z),
                    original_bert_embeddings
                ) * self.config.reconstruction_weight
            else:
                loss_recon = 0.0

            if self.config.identity_weight > 0:
                loss_identity = torch.norm(
                    self.invariant_trans.weight - self.identity
                ) * self.config.identity_weight
            else:
                loss_identity = 0.0

            total_loss = (loss_moral * self.config.moral_weight) + loss_recon + loss_identity

        return SequenceClassifierOutput(loss=total_loss, logits=logits)

In [6]:
def preprocessing(input_text, tokenizer):
    return tokenizer(
        input_text,
        add_special_tokens=True,
        max_length=150,
        padding="max_length",
        truncation=True,
        return_attention_mask=True,
        return_token_type_ids=True,
        return_tensors="pt"
    )

In [7]:
possible_labels = [
    "care", "harm",
    "fairness", "cheating",
    "loyalty", "betrayal",
    "authority", "subversion",
    "purity", "degradation",
    "liberty", "oppression"
]

In [8]:
for lab in possible_labels:
    checkpoint_folder = HF_EXPORT_ROOT / lab.replace(" ", "_")
    print(lab, "->", checkpoint_folder)
    print("exists:", checkpoint_folder.exists(), "| is_dir:", checkpoint_folder.is_dir())

care -> /content/drive/MyDrive/moralbert_inggris/care
exists: True | is_dir: True
harm -> /content/drive/MyDrive/moralbert_inggris/harm
exists: True | is_dir: True
fairness -> /content/drive/MyDrive/moralbert_inggris/fairness
exists: True | is_dir: True
cheating -> /content/drive/MyDrive/moralbert_inggris/cheating
exists: True | is_dir: True
loyalty -> /content/drive/MyDrive/moralbert_inggris/loyalty
exists: True | is_dir: True
betrayal -> /content/drive/MyDrive/moralbert_inggris/betrayal
exists: True | is_dir: True
authority -> /content/drive/MyDrive/moralbert_inggris/authority
exists: True | is_dir: True
subversion -> /content/drive/MyDrive/moralbert_inggris/subversion
exists: True | is_dir: True
purity -> /content/drive/MyDrive/moralbert_inggris/purity
exists: True | is_dir: True
degradation -> /content/drive/MyDrive/moralbert_inggris/degradation
exists: True | is_dir: True
liberty -> /content/drive/MyDrive/moralbert_inggris/liberty
exists: True | is_dir: True
oppression -> /content

In [9]:
import os
import __main__

dummy_py_path = os.path.join(os.getcwd(), "notebook_session.py")
if not os.path.exists(dummy_py_path):
    with open(dummy_py_path, "w", encoding="utf-8") as f:
        f.write("# dummy file for transformers custom model in notebook\n")

__main__.__file__ = dummy_py_path

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

loaded_models = {}

for lab in possible_labels:
    checkpoint_folder = HF_EXPORT_ROOT / lab.replace(" ", "_")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = PlainBERTForSequenceClassification.from_pretrained(
        str(checkpoint_folder),
        local_files_only=True
    ).to(device)

    model.eval()
    loaded_models[lab] = model

print("All models loaded successfully.")
print(checkpoint_folder)
print(os.listdir(checkpoint_folder))

Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/209 [00:01<?, ?it/s]

Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

All models loaded successfully.
/content/drive/MyDrive/moralbert_inggris/oppression
['config.json', 'model.safetensors', 'modeling_plainbert.py', 'configuration_plainbert.py', 'README.md', 'tokenizer_config.json', 'tokenizer.json']


In [11]:
def get_model_score(sentence, label_name):
    model = loaded_models[label_name]

    encodeds = preprocessing(sentence, tokenizer)
    encodeds = {k: v.to(device) for k, v in encodeds.items()}

    with torch.no_grad():
        outputs = model(**encodeds)
        probs = F.softmax(outputs.logits, dim=1)

    score_positive = probs[0, 1].item()
    score_negative = probs[0, 0].item()

    return {
        "score_positive": score_positive,
        "score_negative": score_negative,
        "pred_label": int(score_positive >= 0.5)
    }

In [12]:
test_df = pd.read_csv("test_inggris.csv")
sentences = test_df["sentence"].tolist()

In [13]:
results = []

for sentence in sentences:
    row = {"sentence": sentence}

    for lab in possible_labels:
        output = get_model_score(sentence, lab)
        row[f"{lab}_score"] = output["score_positive"]
        #row[f"{lab}_pred"] = output["pred_label"]

    results.append(row)

results_df = pd.DataFrame(results)
results_df

,sentence,care_score,harm_score,fairness_score,cheating_score,loyalty_score,betrayal_score,authority_score,subversion_score,purity_score,degradation_score,liberty_score,oppression_score
0,After a prolonged Experience he came to know t...,0.001532,0.000859,0.002705,0.000654,0.001519,0.001323,0.001501,0.001309,0.001686,0.004128,0.001036,0.001844
1,That Settles it!,0.001729,0.000751,0.001893,0.000631,0.001721,0.001161,0.001393,0.001103,0.001844,0.003451,0.000976,0.001126
2,3. The Book that runs into a Snarl of Dialect ...,0.001264,0.000997,0.001819,0.000675,0.001670,0.001125,0.001291,0.001122,0.001799,0.004342,0.000820,0.001329
3,Jolly Story of the Slums.,0.002345,0.000816,0.001927,0.000700,0.001541,0.001158,0.001209,0.001075,0.001750,0.004463,0.000881,0.001095
4,etc.,0.001937,0.000840,0.001928,0.000674,0.001622,0.001155,0.001324,0.001075,0.001895,0.003761,0.000884,0.001153
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2486,Once when the gentle rain fell I glided in a b...,0.996725,0.000884,0.001957,0.000679,0.001630,0.001138,0.001252,0.001220,0.002182,0.004521,0.000969,0.001145
2487,"Many times I walked through that valley, and l...",0.001448,0.000892,0.001984,0.000738,0.002311,0.001526,0.001288,0.001310,0.001866,0.004727,0.000884,0.001662
2488,And alway the goal of my fancies was the might...,0.001306,0.000824,0.001901,0.000648,0.001723,0.001112,0.001352,0.001160,0.002384,0.004161,0.000848,0.001315
2489,"After a while, as the days of waking became le...",0.001565,0.000878,0.003848,0.000750,0.010512,0.001920,0.001337,0.001349,0.003420,0.004822,0.002085,0.001472


In [14]:
input_files = test_df["sentence"].values

tokenizer = AutoTokenizer.from_pretrained(base_model)

original_input_id = []
original_attention_masks = []
original_token_type_id = []

def preprocessing(input_text, tokenizer):
    return tokenizer(
        input_text,
        add_special_tokens=True,
        max_length=150,
        padding="max_length",
        truncation=True,
        return_attention_mask=True,
        return_tensors="pt"
    )

for sample in input_files:
    original_encoding_dict = preprocessing(sample, tokenizer)

    original_input_id.append(original_encoding_dict["input_ids"])
    original_attention_masks.append(original_encoding_dict["attention_mask"])

    if "token_type_ids" in original_encoding_dict:
        original_token_type_id.append(original_encoding_dict["token_type_ids"])
    else:
        original_token_type_id.append(torch.zeros_like(original_encoding_dict["input_ids"]))

original_input_id = torch.cat(original_input_id, dim=0)
original_token_type_id = torch.cat(original_token_type_id, dim=0)
original_attention_masks = torch.cat(original_attention_masks, dim=0)

print("recreated")
print("original_input_id.is_meta =", original_input_id.is_meta)
print("original_token_type_id.is_meta =", original_token_type_id.is_meta)
print("original_attention_masks.is_meta =", original_attention_masks.is_meta)

recreated
original_input_id.is_meta = False
original_token_type_id.is_meta = False
original_attention_masks.is_meta = False


In [15]:
possible_labels = [
    "care", "harm",
    "fairness", "cheating",
    "loyalty", "betrayal",
    "authority", "subversion",
    "purity", "degradation",
    "liberty", "oppression"
]

batch_size = 16

torch.set_default_device("cpu")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

predictions = []
threshold_summary = []

for lab_idx, lab in enumerate(possible_labels):
    print(f"\nProcessing label: {lab}")

    best_f1 = 0
    best_th = 0.5
    best_y = None

    true_labels_for_lab = test_df[lab].astype(int).tolist()

    val_set = TensorDataset(
        original_input_id,
        original_token_type_id,
        original_attention_masks,
        torch.tensor(true_labels_for_lab, dtype=torch.long)
    )

    validation_dataloader = DataLoader(val_set, batch_size=batch_size, shuffle=False)

    checkpoint_folder = HF_EXPORT_ROOT / lab.replace(" ", "_")

    model = PlainBERTForSequenceClassification.from_pretrained(
        str(checkpoint_folder),
        local_files_only=True
    )
    model = model.to(device)
    model.eval()

    all_probs = []
    y_true = []

    for batch in validation_dataloader:
        b_input_ids, b_token_type_ids, b_input_mask, b_labels = [
            t.to(device) for t in batch
        ]

        model_inputs = {
            "input_ids": b_input_ids,
            "attention_mask": b_input_mask
        }

        if b_token_type_ids is not None:
            model_inputs["token_type_ids"] = b_token_type_ids

        with torch.no_grad():
            outputs = model(**model_inputs)

            logits = outputs.logits.detach().cpu()
            probs = torch.softmax(logits, dim=1).numpy()[:, 1]

            all_probs.extend(probs.tolist())
            y_true.extend(b_labels.detach().cpu().numpy().tolist())

    all_probs = np.array(all_probs)
    y_true = np.array(y_true)

    for th in np.arange(0.05, 1.00, 0.05):
        y_pred = (all_probs >= th).astype(int)
        f1 = f1_score(y_true, y_pred, average="binary", zero_division=0)

        if f1 > best_f1:
            best_f1 = f1
            best_th = th
            best_y = y_pred.copy()

    threshold_summary.append({
        "label": lab,
        "best_threshold": float(best_th),
        "best_f1": float(best_f1)
    })

    if lab_idx == 0:
        for ex_id, (pred_label, true_label) in enumerate(zip(best_y, y_true)):
            predictions.append({
                "id": ex_id,
                f"pred_{lab}": int(pred_label),
                f"true_{lab}": int(true_label)
            })
    else:
        for ex_id, (pred_label, true_label) in enumerate(zip(best_y, y_true)):
            predictions[ex_id][f"pred_{lab}"] = int(pred_label)
            predictions[ex_id][f"true_{lab}"] = int(true_label)

    print("Evaluation")
    print(f"{lab} | best threshold: {best_th:.2f} | best F1: {best_f1:.4f}")

    target_names = [f"Non-{lab}", lab]
    report = classification_report(
        y_true,
        best_y,
        target_names=target_names,
        zero_division=0
    )

    print("\nClassification Report:")
    print(report)

pred_df = pd.DataFrame(predictions)
threshold_df = pd.DataFrame(threshold_summary)


Processing label: care


Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

Evaluation
care | best threshold: 0.95 | best F1: 0.9781

Classification Report:
              precision    recall  f1-score   support

    Non-care       1.00      1.00      1.00      2355
        care       0.97      0.99      0.98       136

    accuracy                           1.00      2491
   macro avg       0.99      0.99      0.99      2491
weighted avg       1.00      1.00      1.00      2491


Processing label: harm


Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

Evaluation
harm | best threshold: 0.90 | best F1: 0.9533

Classification Report:
              precision    recall  f1-score   support

    Non-harm       1.00      1.00      1.00      2383
        harm       0.96      0.94      0.95       108

    accuracy                           1.00      2491
   macro avg       0.98      0.97      0.98      2491
weighted avg       1.00      1.00      1.00      2491


Processing label: fairness


...

Evaluation
degradation | best threshold: 0.75 | best F1: 0.8571

Classification Report:
                 precision    recall  f1-score   support

Non-degradation       1.00      1.00      1.00      2483
    degradation       1.00      0.75      0.86         8

       accuracy                           1.00      2491
      macro avg       1.00      0.88      0.93      2491
   weighted avg       1.00      1.00      1.00      2491


Processing label: liberty


Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

Evaluation
liberty | best threshold: 0.20 | best F1: 0.9362

Classification Report:
              precision    recall  f1-score   support

 Non-liberty       1.00      1.00      1.00      2468
     liberty       0.92      0.96      0.94        23

    accuracy                           1.00      2491
   macro avg       0.96      0.98      0.97      2491
weighted avg       1.00      1.00      1.00      2491


Processing label: oppression


Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

Evaluation
oppression | best threshold: 0.80 | best F1: 0.9375

Classification Report:
                precision    recall  f1-score   support

Non-oppression       1.00      1.00      1.00      2475
    oppression       0.94      0.94      0.94        16

      accuracy                           1.00      2491
     macro avg       0.97      0.97      0.97      2491
  weighted avg       1.00      1.00      1.00      2491



In [16]:
pd.set_option('display.max_columns', None)

In [17]:
results = []

for idx_lab, lab in enumerate(possible_labels):
    result = {"Moral Value": lab}
    true = test_df[lab].values
    candidate = pred_df["pred_" + lab].values

    result["F1 Score (Binary)"] = f1_score(true, candidate, average="binary", zero_division=0)
    result["F1 Score (Weighted)"] = f1_score(true, candidate, average="weighted", zero_division=0)

    result["Precision Score (Binary)"] = precision_score(true, candidate, average="binary", zero_division=0)
    result["Precision Score (Weighted)"] = precision_score(true, candidate, average="weighted", zero_division=0)

    result["Recall Score (Binary)"] = recall_score(true, candidate, average="binary", zero_division=0)
    result["Recall Score (Weighted)"] = recall_score(true, candidate, average="weighted", zero_division=0)

    result["Accuracy"] = accuracy_score(true, candidate)

    results.append(result)

results = pd.DataFrame(results)

In [18]:
possible_labels = [
    "care", "harm",
    "fairness", "cheating",
    "loyalty", "betrayal",
    "authority", "subversion",
    "purity", "degradation",
    "liberty", "oppression"
]

test_df.reset_index(drop=True, inplace=True)
n_bootstrap_iters = 1000

bootstrap_results = {
    label: {
        metric: [] for metric in [
            "F1 (Binary)", "F1 (Macro)", "F1 (Weighted)",
            "Precision (Binary)", "Precision (Macro)", "Precision (Weighted)",
            "Recall (Binary)", "Recall (Macro)", "Recall (Weighted)",
            "Accuracy"
        ]
    } for label in possible_labels
}

for _ in range(n_bootstrap_iters):
    for lab in possible_labels:
        sample_indices = resample(np.arange(len(test_df)), replace=True)
        true = test_df.loc[sample_indices, lab].values
        candidate = pred_df.loc[sample_indices, f"pred_{lab}"].values

        bootstrap_results[lab]["F1 (Binary)"].append(f1_score(true, candidate, average="binary", zero_division=0))
        bootstrap_results[lab]["F1 (Macro)"].append(f1_score(true, candidate, average="macro", zero_division=0))
        bootstrap_results[lab]["F1 (Weighted)"].append(f1_score(true, candidate, average="weighted", zero_division=0))
        bootstrap_results[lab]["Precision (Binary)"].append(precision_score(true, candidate, average="binary", zero_division=0))
        bootstrap_results[lab]["Precision (Macro)"].append(precision_score(true, candidate, average="macro", zero_division=0))
        bootstrap_results[lab]["Precision (Weighted)"].append(precision_score(true, candidate, average="weighted", zero_division=0))
        bootstrap_results[lab]["Recall (Binary)"].append(recall_score(true, candidate, average="binary", zero_division=0))
        bootstrap_results[lab]["Recall (Macro)"].append(recall_score(true, candidate, average="macro", zero_division=0))
        bootstrap_results[lab]["Recall (Weighted)"].append(recall_score(true, candidate, average="weighted", zero_division=0))
        bootstrap_results[lab]["Accuracy"].append(accuracy_score(true, candidate))

std_devs = {
    label: {metric: np.std(values) for metric, values in metrics.items()}
    for label, metrics in bootstrap_results.items()
}

final_results = []
for lab in possible_labels:
    result = {"Moral Value": lab}
    true = test_df[lab].values
    candidate = pred_df[f"pred_{lab}"].values

    result["F1 Score (Binary)"] = f"{f1_score(true, candidate, average='binary', zero_division=0):.2f} ± {std_devs[lab]['F1 (Binary)']:.2f}"
    result["F1 Score (Macro)"] = f"{f1_score(true, candidate, average='macro', zero_division=0):.2f} ± {std_devs[lab]['F1 (Macro)']:.2f}"
    result["F1 Score (Weighted)"] = f"{f1_score(true, candidate, average='weighted', zero_division=0):.2f} ± {std_devs[lab]['F1 (Weighted)']:.2f}"

    result["Precision Score (Binary)"] = f"{precision_score(true, candidate, average='binary', zero_division=0):.2f} ± {std_devs[lab]['Precision (Binary)']:.2f}"
    result["Precision Score (Macro)"] = f"{precision_score(true, candidate, average='macro', zero_division=0):.2f} ± {std_devs[lab]['Precision (Macro)']:.2f}"
    result["Precision Score (Weighted)"] = f"{precision_score(true, candidate, average='weighted', zero_division=0):.2f} ± {std_devs[lab]['Precision (Weighted)']:.2f}"

    result["Recall Score (Binary)"] = f"{recall_score(true, candidate, average='binary', zero_division=0):.2f} ± {std_devs[lab]['Recall (Binary)']:.2f}"
    result["Recall Score (Macro)"] = f"{recall_score(true, candidate, average='macro', zero_division=0):.2f} ± {std_devs[lab]['Recall (Macro)']:.2f}"
    result["Recall Score (Weighted)"] = f"{recall_score(true, candidate, average='weighted', zero_division=0):.2f} ± {std_devs[lab]['Recall (Weighted)']:.2f}"

    result["Accuracy"] = f"{accuracy_score(true, candidate):.2f} ± {std_devs[lab]['Accuracy']:.2f}"

    final_results.append(result)

results_df = pd.DataFrame(final_results)
results_df

,Moral Value,F1 Score (Binary),F1 Score (Macro),F1 Score (Weighted),Precision Score (Binary),Precision Score (Macro),Precision Score (Weighted),Recall Score (Binary),Recall Score (Macro),Recall Score (Weighted),Accuracy
0,care,0.98 ± 0.01,0.99 ± 0.00,1.00 ± 0.00,0.97 ± 0.01,0.99 ± 0.01,1.00 ± 0.00,0.99 ± 0.01,0.99 ± 0.01,1.00 ± 0.00,1.00 ± 0.00
1,harm,0.95 ± 0.01,0.98 ± 0.01,1.00 ± 0.00,0.96 ± 0.02,0.98 ± 0.01,1.00 ± 0.00,0.94 ± 0.02,0.97 ± 0.01,1.00 ± 0.00,1.00 ± 0.00
2,fairness,0.95 ± 0.02,0.97 ± 0.01,1.00 ± 0.00,0.98 ± 0.02,0.99 ± 0.01,1.00 ± 0.00,0.91 ± 0.03,0.96 ± 0.02,1.00 ± 0.00,1.00 ± 0.00
3,cheating,0.85 ± 0.06,0.92 ± 0.03,1.00 ± 0.00,0.81 ± 0.09,0.90 ± 0.04,1.00 ± 0.00,0.89 ± 0.07,0.95 ± 0.04,1.00 ± 0.00,1.00 ± 0.00
4,loyalty,0.80 ± 0.06,0.90 ± 0.03,0.99 ± 0.00,0.89 ± 0.06,0.94 ± 0.03,0.99 ± 0.00,0.73 ± 0.08,0.86 ± 0.04,1.00 ± 0.00,1.00 ± 0.00
5,betrayal,0.89 ± 0.10,0.94 ± 0.05,1.00 ± 0.00,0.89 ± 0.12,0.94 ± 0.06,1.00 ± 0.00,0.89 ± 0.12,0.94 ± 0.06,1.00 ± 0.00,1.00 ± 0.00
6,authority,0.99 ± 0.01,0.99 ± 0.00,1.00 ± 0.00,0.98 ± 0.01,0.99 ± 0.01,1.00 ± 0.00,0.99 ± 0.01,1.00 ± 0.00,1.00 ± 0.00,1.00 ± 0.00
7,subversion,0.73 ± 0.18,0.86 ± 0.09,1.00 ± 0.00,1.00 ± 0.14,1.00 ± 0.07,1.00 ± 0.00,0.57 ± 0.20,0.79 ± 0.10,1.00 ± 0.00,1.00 ± 0.00
8,purity,0.95 ± 0.04,0.97 ± 0.02,1.00 ± 0.00,0.95 ± 0.05,0.97 ± 0.02,1.00 ± 0.00,0.95 ± 0.05,0.97 ± 0.03,1.00 ± 0.00,1.00 ± 0.00
9,degradation,0.86 ± 0.12,0.93 ± 0.06,1.00 ± 0.00,1.00 ± 0.04,1.00 ± 0.02,1.00 ± 0.00,0.75 ± 0.17,0.88 ± 0.08,1.00 ± 0.00,1.00 ± 0.00
